# FinDER local LangChain RAG baseline

This notebook uses **free local inference**: FinDER reference passages → Hugging Face sentence-transformer embeddings → MMR retrieval → Ollama/Llama 3.2 answer with source labels. No OpenAI API key or credits are used.

> Before running: install [Ollama for Windows](https://ollama.com/download/windows), open the Ollama app, and leave it running. The first run downloads models and the dataset.

## 1. Install the Python dependencies
This installs into the currently selected Jupyter kernel. Restart the kernel if Jupyter asks you to.

In [ ]:
# %pip install --upgrade -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


## 2. Check Ollama and download the local chat model
The default `llama3.2` model is about 2 GB. Ollama skips the download if it is already present.

In [2]:
import os
import shutil
import subprocess
import time
from pathlib import Path

from dotenv import load_dotenv
from rag import RAGSettings, check_ollama

load_dotenv()
settings = RAGSettings(
    sample_size=300,
    top_k=4,
    chat_model=os.getenv("OLLAMA_MODEL", "llama3.2"),
    ollama_base_url=os.getenv("OLLAMA_BASE_URL", "http://localhost:11434"),
    embedding_model=os.getenv(
        "HF_EMBEDDING_MODEL", "sentence-transformers/all-MiniLM-L6-v2"
    ),
    embedding_device=os.getenv("HF_EMBEDDING_DEVICE", "cpu"),
)

portable_root = Path(r"C:\Users\20122\Documents\Codex\Ollama")
ollama_exe = shutil.which("ollama") or portable_root / "ollama.exe"
if not Path(ollama_exe).is_file():
    raise RuntimeError(
        "Ollama was not found. Install it from https://ollama.com/download/windows."
    )

ollama_env = os.environ.copy()
if portable_root.exists():
    ollama_env["OLLAMA_MODELS"] = str(portable_root / "models")
    ollama_env["USERPROFILE"] = str(portable_root)

ollama_status = check_ollama(settings.ollama_base_url, settings.chat_model)
if "not reachable" in ollama_status.message:
    creation_flags = subprocess.CREATE_NO_WINDOW if os.name == "nt" else 0
    subprocess.Popen(
        [str(ollama_exe), "serve"],
        env=ollama_env,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        creationflags=creation_flags,
    )
    for _ in range(30):
        time.sleep(1)
        ollama_status = check_ollama(settings.ollama_base_url, settings.chat_model)
        if "not reachable" not in ollama_status.message:
            break

if not ollama_status.ready:
    subprocess.run(
        [str(ollama_exe), "pull", settings.chat_model], env=ollama_env, check=True
    )
    ollama_status = check_ollama(settings.ollama_base_url, settings.chat_model)
assert ollama_status.ready, ollama_status.message
print(ollama_status.message)

Ollama is ready with `llama3.2`.


## 3. Load a deterministic FinDER sample

In [3]:
from rag import load_finder_records, records_to_documents, split_documents

records = load_finder_records(settings.sample_size, settings.seed)
documents = records_to_documents(records)
chunks = split_documents(documents, settings.chunk_size, settings.chunk_overlap)

print(f"Records: {len(records):,}")
print(f"Reference passages: {len(documents):,}")
print(f"Chunks: {len(chunks):,}")

Records: 300
Reference passages: 321
Chunks: 1,280


## 4. Build the local RAG index
The first run downloads the MiniLM embedding model. Embeddings are computed locally, so this cell cannot incur API charges.

In [4]:
from rag import build_rag

rag, stats = build_rag(settings=settings, records=records)
stats

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\Users\20122\anaconda3\Lib\site-packages\huggingface_hub\file_download.py:142: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\20122\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

{'records': 300, 'references': 321, 'chunks': 1280}

## 5. Ask a grounded question

In [5]:
response = rag.ask("What financial risks or changes are described in the indexed sample?")
print(response.answer)

for number, source in enumerate(response.sources, start=1):
    print(f"\n--- S{number} | row {source.metadata.get('row_id')} ---")
    print(source.page_content[:700])

According to the retrieved passages, the indexed sample describes the following financial risks or changes:

1. Increases in FTP credits on deposits allocated to the business segments, which led to a decrease in net interest income on an FTE basis. [S2]
2. Increases in interest expense on long-term debt and deposits, which also contributed to the decrease in net interest income. [S2]
3. Decreases in interest income on loans and leases, which further reduced net interest income. [S2]
4. Increases in FTP charges to the business segments on loans and leases, which partially offset the negative impacts. [S2]
5. Increases in interest income on investment securities and other short-term investments, which also partially offset the negative impacts. [S2]
6. Changes in market interest rates, which drove the increases in FTP credits and FTP charges allocated to the business segments. [S2]
7. Net sales and maturities of available-for sale securities, which led to an increase in cash provided by 